# Identify Energy Generator Locations

## Identify location weight for weather data aggregation

* Source: https://www.marktstammdatenregister.de/MaStR/
	- under Öffentlichen Daten
	- Filter
		- Betriebstatus = In Betrieb
		- Energieträger = Solarstrahlungsenergie or Wind
		- Bruttoleistung der Einheiten > 5000
* download csv of actual market data - energy generator register 
* identify the locations and sorted by capacitys for wind and solar generation
	- large PV parks aggregate the solor generation
	- wind parks registered as small units up to 15000
	- identify geo-cluster of the largest generators
	- center location of clusers and sum of capacity in the cluster 
* use cluster centers as selected locations for weather data search
* use capacity sum in the cluster to weight the weather data


### Load and explore the csv

* visualize the generator distribution on map of Germany
* find clusters using K-Means 
* create dictionary of coordinates and aggregated (sum of) capacities
* visualize the cluster distribution on map of Germany

In [1]:
import pandas as pd

df_gen_pv = pd.read_csv("../data/raw/Stromerzeuger_pv_20260530.csv", sep=";", decimal=",")
df_gen_wind = pd.read_csv("../data/raw/Stromerzeuger_wind_20260530.csv", sep=";", decimal=",")

# select relevant columns
cols_pv =['Anzeige-Name der Einheit', 
        'Bruttoleistung der Einheit', 
        'Nettonennleistung der Einheit', 
        'Inbetriebnahmedatum der Einheit', 
        'Bundesland', 'Postleitzahl', 'Ort', 
        'Koordinate: Breitengrad (WGS84)', 'Koordinate: Längengrad (WGS84)'
        ]

cols_wind =['Anzeige-Name der Einheit', 
        'Bruttoleistung der Einheit', 
        'Nettonennleistung der Einheit', 
        'Inbetriebnahmedatum der Einheit', 
        'Bundesland', 'Postleitzahl', 'Ort', 
        'Koordinate: Breitengrad (WGS84)', 'Koordinate: Längengrad (WGS84)', 
        'Wind an Land oder auf See'
        #'Name des Windparks', 'Elektrische KWK-Leistung'
        ]

df_gen_pv = df_gen_pv[cols_pv]
df_gen_wind = df_gen_wind[cols_wind]

print("--- Solar ---")
display(df_gen_pv.info())
display(df_gen_pv.describe())
df_gen_pv = df_gen_pv.sort_values('Bruttoleistung der Einheit').reset_index(drop=True)
df_gen_pv.head(10)

print("--- Wind ---")
display(df_gen_wind.info())
display(df_gen_wind.describe())
df_gen_wind = df_gen_wind.sort_values('Bruttoleistung der Einheit').reset_index(drop=True)
df_gen_wind.head(10)
        
        
print('solar missing:\n', df_gen_pv.isna().sum())
print('wind missing:\n', df_gen_wind.isna().sum())

--- Solar ---
<class 'pandas.DataFrame'>
RangeIndex: 2363 entries, 0 to 2362
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Anzeige-Name der Einheit         2363 non-null   str    
 1   Bruttoleistung der Einheit       2363 non-null   float64
 2   Nettonennleistung der Einheit    2363 non-null   float64
 3   Inbetriebnahmedatum der Einheit  2363 non-null   str    
 4   Bundesland                       2358 non-null   str    
 5   Postleitzahl                     2363 non-null   int64  
 6   Ort                              2363 non-null   str    
 7   Koordinate: Breitengrad (WGS84)  2359 non-null   float64
 8   Koordinate: Längengrad (WGS84)   2359 non-null   float64
dtypes: float64(4), int64(1), str(4)
memory usage: 288.1 KB


None

,Bruttoleistung der Einheit,Nettonennleistung der Einheit,Postleitzahl,Koordinate: Breitengrad (WGS84),Koordinate: Längengrad (WGS84)
count,2363.000000,2363.000000,2363.000000,2359.000000,2359.000000
mean,11915.140958,10202.895713,51153.619975,50.998376,11.107136
std,11603.642489,9809.342240,35093.244205,1.925011,1.988913
min,5000.040000,5.000000,1109.000000,47.639908,6.079548
25%,6360.870000,5500.000000,17172.500000,49.358143,9.865538
50%,8852.700000,7525.000000,47495.000000,50.912559,11.374787
75%,12590.940000,10836.075000,89341.500000,52.568602,12.441681
max,162262.200000,144000.000000,99996.000000,57.268518,15.071009


--- Wind ---
<class 'pandas.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Anzeige-Name der Einheit         2919 non-null   str    
 1   Bruttoleistung der Einheit       2919 non-null   int64  
 2   Nettonennleistung der Einheit    2919 non-null   int64  
 3   Inbetriebnahmedatum der Einheit  2919 non-null   str    
 4   Bundesland                       2919 non-null   str    
 5   Postleitzahl                     1904 non-null   float64
 6   Ort                              1904 non-null   str    
 7   Koordinate: Breitengrad (WGS84)  2919 non-null   float64
 8   Koordinate: Längengrad (WGS84)   2919 non-null   float64
 9   Wind an Land oder auf See        2919 non-null   str    
dtypes: float64(3), int64(2), str(5)
memory usage: 410.6 KB


None

,Bruttoleistung der Einheit,Nettonennleistung der Einheit,Postleitzahl,Koordinate: Breitengrad (WGS84),Koordinate: Längengrad (WGS84)
count,2919.000000,2919.000000,1904.000000,2919.000000,2919.000000
mean,6488.562521,6488.562521,33585.596113,53.016829,9.330003
std,1594.947520,1594.947520,18518.447386,1.473616,2.663446
min,5075.000000,5075.000000,1612.000000,47.886036,4.943528
25%,5600.000000,5600.000000,21734.000000,51.958167,6.951990
50%,6000.000000,6000.000000,29386.000000,53.398539,9.003209
75%,6465.000000,6465.000000,45713.500000,54.160154,11.027549
max,15000.000000,15000.000000,99947.000000,54.888930,14.733678


solar missing:
 Anzeige-Name der Einheit           0
Bruttoleistung der Einheit         0
Nettonennleistung der Einheit      0
Inbetriebnahmedatum der Einheit    0
Bundesland                         5
Postleitzahl                       0
Ort                                0
Koordinate: Breitengrad (WGS84)    4
Koordinate: Längengrad (WGS84)     4
dtype: int64
wind missing:
 Anzeige-Name der Einheit              0
Bruttoleistung der Einheit            0
Nettonennleistung der Einheit         0
Inbetriebnahmedatum der Einheit       0
Bundesland                            0
Postleitzahl                       1015
Ort                                1015
Koordinate: Breitengrad (WGS84)       0
Koordinate: Längengrad (WGS84)        0
Wind an Land oder auf See             0
dtype: int64


In [2]:
# show missing coordinates with bundesland and ort
missing_coords_pv = df_gen_pv[df_gen_pv['Koordinate: Breitengrad (WGS84)'].isna() | df_gen_pv['Koordinate: Längengrad (WGS84)'].isna()]
print(f"Anzahl fehlender Koordinaten bei PV: {len(missing_coords_pv)}")
display(missing_coords_pv[['Anzeige-Name der Einheit', 'Bundesland', 'Ort']])

missing_coords_wind = df_gen_wind[df_gen_wind['Koordinate: Breitengrad (WGS84)'].isna() | df_gen_wind['Koordinate: Längengrad (WGS84)'].isna()]
print(f"Anzahl fehlender Koordinaten bei Wind: {len(missing_coords_wind)}")
display(missing_coords_wind[['Anzeige-Name der Einheit', 'Bundesland', 'Ort']])

coordinates_pv_missing_dict = {'Freiburg': (47.9990, 7.8421), 
                               'Dolgesheim': (49.8467, 8.2056), 
                               'Solingen': (51.1657, 7.0116), 
                               'Sassenberg': (51.9000, 8.3167),
                               'Friedeburg': (53.5000, 7.5000),}
# replace missing coordinates in df_gen_pv
for idx, row in df_gen_pv.iterrows():
    if pd.isna(row['Koordinate: Breitengrad (WGS84)']) or pd.isna(row['Koordinate: Längengrad (WGS84)']):
        ort = row['Ort']
        if ort in coordinates_pv_missing_dict:
            lat, lon = coordinates_pv_missing_dict[ort]
            df_gen_pv.at[idx, 'Koordinate: Breitengrad (WGS84)'] = lat
            df_gen_pv.at[idx, 'Koordinate: Längengrad (WGS84)'] = lon

print('solar missing after imputation:\n', df_gen_pv.isna().sum())

Anzahl fehlender Koordinaten bei PV: 4


,Anzeige-Name der Einheit,Bundesland,Ort
1012,Photovoltaikanlage Enpal,Niedersachsen,Friedeburg
1118,Beckers-Sonnendach,Rheinland-Pfalz,Dolgesheim
1223,geraniensolar,Nordrhein-Westfalen,Solingen
1942,Vom-Stein-Str. 2,Nordrhein-Westfalen,Sassenberg


Anzahl fehlender Koordinaten bei Wind: 0


,Anzeige-Name der Einheit,Bundesland,Ort


solar missing after imputation:
 Anzeige-Name der Einheit           0
Bruttoleistung der Einheit         0
Nettonennleistung der Einheit      0
Inbetriebnahmedatum der Einheit    0
Bundesland                         5
Postleitzahl                       0
Ort                                0
Koordinate: Breitengrad (WGS84)    0
Koordinate: Längengrad (WGS84)     0
dtype: int64


In [ ]:
df_gen_pv.to_csv("../data/processed/pv_plants.csv", index=False)
df_gen_wind.to_csv("../data/processed/wind_plants.csv", index=False)

* OpenStreetMap (default) blocks the local display
* use alternative tiles: 
    - tiles="OpenStreetMap" (führt zu Blockierung)
    - tiles="CartoDB positron"
    - tiles="CartoDB dark_matter"

In [3]:
# use KMeans clustering to find the top 10 clusters of solar plants and their centroids
from shap import kmeans
from sklearn.cluster import KMeans  
import numpy as np
import folium

kmeans_pv = KMeans(n_clusters=10, random_state=42)
X = df_gen_pv[['Koordinate: Breitengrad (WGS84)', 'Koordinate: Längengrad (WGS84)']]
kmeans_pv.fit(X)
centroids = kmeans_pv.cluster_centers_
print("Top 10 clusters of solar plants (centroids):")
for i, centroid in enumerate(centroids):
    print(f"Cluster {i+1}: Latitude {centroid[0]:.4f}, Longitude {centroid[1]:.4f}")

# show the top 10 clusters of solar plants and their centroids and the solar plants belonging to each cluster on a map
m = folium.Map(location=[51.1657, 10.4515], 
               zoom_start=6, 
               tiles="CartoDB positron")    
for _, row in df_gen_pv[df_gen_pv['Nettonennleistung der Einheit'] > 100].iterrows():
    folium.Marker(
        location=[row['Koordinate: Breitengrad (WGS84)'], row['Koordinate: Längengrad (WGS84)']],
        popup=f"{row['Anzeige-Name der Einheit']} ({row['Nettonennleistung der Einheit']} MW)"
    ).add_to(m) 
for i, centroid in enumerate(centroids):
    folium.Marker(
        location=[centroid[0], centroid[1]],
        popup=f"Cluster {i+1} Centroid",
        icon=folium.Icon(color='red', icon='star')
    ).add_to(m) 
m.save("../reports/solar_clusters_map.html")


Top 10 clusters of solar plants (centroids):
Cluster 1: Latitude 48.2542, Longitude 9.9548
Cluster 2: Latitude 53.2119, Longitude 12.0865
Cluster 3: Latitude 50.0866, Longitude 7.0443
Cluster 4: Latitude 51.5458, Longitude 13.8070
Cluster 5: Latitude 48.9839, Longitude 11.9081
Cluster 6: Latitude 49.9259, Longitude 10.2570
Cluster 7: Latitude 54.1992, Longitude 9.6420
Cluster 8: Latitude 52.3168, Longitude 8.4481
Cluster 9: Latitude 53.1420, Longitude 13.7043
Cluster 10: Latitude 51.5413, Longitude 11.7167


In [7]:
# create a dataframe with the top 10 clusters of solar plants and their centroids and the total capacity of the plants in each cluster
clusters = kmeans_pv.labels_
df_pv_clusters = pd.DataFrame({
    'Cluster': clusters,
    'Latitude': df_gen_pv['Koordinate: Breitengrad (WGS84)'],
    'Longitude': df_gen_pv['Koordinate: Längengrad (WGS84)'],
    'Nettonennleistung der Einheit': df_gen_pv['Nettonennleistung der Einheit']
})
df_pv_clusters = df_pv_clusters.groupby('Cluster').agg({
    'Latitude': 'mean',
    'Longitude': 'mean',
    'Nettonennleistung der Einheit': 'sum'
}).reset_index()
df_pv_clusters = df_pv_clusters.sort_values(by='Nettonennleistung der Einheit', ascending=False)
print("Top 10 clusters of solar plants with total capacity:")
display(df_pv_clusters)

# show the cluster centroids and the total capacity of the plants in each cluster on a map with the size of the marker proportional to the total capacity
m = folium.Map(location=[51.1657, 10.4515], 
                zoom_start=6, 
                tiles="CartoDB positron")
for _, row in df_pv_clusters.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=row['Nettonennleistung der Einheit'] / 100000,  # adjust the radius for better visualization
        popup=f"Cluster {row['Cluster']} - Total Capacity: {row['Nettonennleistung der Einheit']} MW",
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(m)
m.save("../reports/solar_clusters_capacity_map.html")


Top 10 clusters of solar plants with total capacity:


,Cluster,Latitude,Longitude,Nettonennleistung der Einheit
4,4,48.983946,11.908091,3771865.669
5,5,49.925902,10.256972,3054019.177
1,1,53.208643,12.085283,2915049.393
8,8,53.141955,13.704316,2898553.027
9,9,51.537985,11.716391,2890191.336
3,3,51.545801,13.807043,2432306.107
2,2,50.086581,7.044273,1850665.891
6,6,54.199199,9.642025,1846665.265
0,0,48.254208,9.954836,1577292.640
7,7,52.316794,8.448149,872834.065


In [8]:
# save the top 10 clusters of solar plants with total capacity to a csv file
df_pv_clusters.to_csv("../data/processed/pv_clusters.csv", index=False)

In [5]:
# use KMeans clustering to find the top 10 clusters of wind plants and their centroids and show the top 10 clusters of wind plants and their centroids and the wind plants belonging to each cluster on a map
kmeans_wind = KMeans(n_clusters=10, random_state=42)
X = df_gen_wind[['Koordinate: Breitengrad (WGS84)', 'Koordinate: Längengrad (WGS84)']]
kmeans_wind.fit(X)
centroids = kmeans_wind.cluster_centers_
print("Top 10 clusters of wind plants (centroids):")
for i, centroid in enumerate(centroids):
    print(f"Cluster {i+1}: Latitude {centroid[0]:.4f}, Longitude {centroid[1]:.4f}")
m = folium.Map(location=[51.1657, 10.4515], 
                zoom_start=6, 
                tiles="CartoDB positron"
               )    
for _, row in df_gen_wind[df_gen_wind['Nettonennleistung der Einheit'] > 100].iterrows():
    folium.Marker(
        location=[row['Koordinate: Breitengrad (WGS84)'], row['Koordinate: Längengrad (WGS84)']],
        popup=f"{row['Anzeige-Name der Einheit']} ({row['Nettonennleistung der Einheit']} MW)"
    ).add_to(m)
for i, centroid in enumerate(centroids):
    folium.Marker(
        location=[centroid[0], centroid[1]],
        popup=f"Cluster {i+1} Centroid",
        icon=folium.Icon(color='red', icon='star')
    ).add_to(m)
m.save("../reports/wind_clusters_map.html")


Top 10 clusters of wind plants (centroids):
Cluster 1: Latitude 51.8430, Longitude 8.4977
Cluster 2: Latitude 54.7202, Longitude 13.9190
Cluster 3: Latitude 53.9376, Longitude 7.2918
Cluster 4: Latitude 52.2084, Longitude 10.4919
Cluster 5: Latitude 54.0124, Longitude 9.7626
Cluster 6: Latitude 52.4380, Longitude 13.9644
Cluster 7: Latitude 50.7704, Longitude 6.9432
Cluster 8: Latitude 54.2163, Longitude 6.2127
Cluster 9: Latitude 48.8205, Longitude 9.7377
Cluster 10: Latitude 52.4746, Longitude 12.1500


In [6]:
# create a dataframe with the top 10 clusters of wind plants and their centroids and the total capacity of the plants in each cluster
clusters = kmeans_wind.labels_
df_wind_clusters = pd.DataFrame({
    'Cluster': clusters,
    'Latitude': df_gen_wind['Koordinate: Breitengrad (WGS84)'],
    'Longitude': df_gen_wind['Koordinate: Längengrad (WGS84)'],
    'Nettonennleistung der Einheit': df_gen_wind['Nettonennleistung der Einheit']
})
df_wind_clusters = df_wind_clusters.groupby('Cluster').agg({
    'Latitude': 'mean',
    'Longitude': 'mean',
    'Nettonennleistung der Einheit': 'sum'
}).reset_index()
print("Top 10 clusters of wind plants with total capacity:")
display(df_wind_clusters)

# show the cluster centroids and the total capacity of the plants in each cluster on a map with the size of the marker proportional to the total capacity
m = folium.Map(location=[51.1657, 10.4515], 
                zoom_start=6, 
                tiles="CartoDB positron"
               )
for _, row in df_wind_clusters.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=row['Nettonennleistung der Einheit'] / 100000,  # adjust the radius for better visualization
        popup=f"Cluster {row['Cluster']} - Total Capacity: {row['Nettonennleistung der Einheit']} MW",
        color='green',
        fill=True,
        fill_color='green'
    ).add_to(m)
m.save("../reports/wind_clusters_capacity_map.html")


Top 10 clusters of wind plants with total capacity:


,Cluster,Latitude,Longitude,Nettonennleistung der Einheit
0,0,51.843002,8.497739,2175340
1,1,54.720161,13.918991,1624205
2,2,53.937550,7.291752,2381598
3,3,52.208403,10.491896,1608100
4,4,54.012429,9.762561,2414130
5,5,52.437985,13.964374,1272560
6,6,50.770402,6.943235,1383920
7,7,54.216299,6.212748,4220760
8,8,48.820484,9.737655,419120
9,9,52.474595,12.150012,1440381


In [9]:
# save the top 10 clusters of wind plants with total capacity to a csv file
df_wind_clusters.to_csv("../data/processed/wind_clusters.csv", index=False)